# YOLOv1 — VOC2007 training and ZIP export

Attach **stpeteishii/pascal-voc-2007-dataset** via Add Input. Select **GPU T4 ×2** and enable Internet. Training uses two DDP processes (one per GPU).
Remote repository root must match `paper/Personal_Task` (it contains `code/`).
Run `smoke`, then `overfit`, then `train`. Only edit the settings cell.
Weights initialize **16** conv layers; the binary training history is not independently verified.
This dataset was checked to contain train/val/trainval; a direct test.txt request returned 404. Results default to validation.


In [ ]:
REPO_URL = "https://github.com/Dacekey/YOLOv1_PersonalTask_DPL302m.git"
# Set an immutable full commit hash after pushing. A branch/tag is accepted for initial smoke only.
REPO_REF = "main"
RUN_MODE = "smoke"  # smoke | overfit | train
DATASET_DIR = ""     # empty: automatically locate exactly one VOC2007 root
PRETRAINED_PATH = "code/pretrain/extraction.conv.weights" # relative to cloned REPO root; absolute attached paths also work
RESUME_PATH = ""     # last.pt from an attached previous output; also attach sibling best.pt if available
MAX_HOURS = 9.0      # includes setup, with export reserve; set below your session limit
RUN_NAME = "yolov1_voc2007_smoke_01"  # choose a NEW name for each fresh run


In [ ]:
import importlib.util
import pathlib
import subprocess
import sys
import time
SESSION_START = time.monotonic()
assert RUN_MODE in {"smoke", "overfit", "train"}
assert "YOUR_USER" not in REPO_URL, "Set REPO_URL first"
assert RUN_NAME and pathlib.Path(RUN_NAME).name == RUN_NAME, "RUN_NAME must be a directory name"
import torch
assert torch.cuda.is_available() and torch.cuda.device_count() == 2, "Select GPU T4 ×2 in Kaggle settings"
print("PyTorch:", torch.__version__, "GPUs:", [torch.cuda.get_device_name(i) for i in range(2)])
# Keep Kaggle's CUDA PyTorch. Do not install the local CPU requirements.lock.txt.
missing = [pkg for module, pkg in [("PIL", "pillow>=10,<13"), ("matplotlib", "matplotlib>=3.8,<4"), ("numpy", "numpy>=1.26,<3")] if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
WORK = pathlib.Path("/kaggle/working")
WORK.mkdir(parents=True, exist_ok=True)
# Source stays outside /kaggle/working so saved output contains only the run and ZIP.
REPO = pathlib.Path("/kaggle/temp/personal_yolov1_repo")
REPO.parent.mkdir(parents=True, exist_ok=True)
if not (REPO / ".git").is_dir():
    assert not REPO.exists(), f"Non-git path already exists: {REPO}"
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(REPO)], check=True)
else:
    origin = subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip()
    assert origin == REPO_URL, "Existing checkout has a different origin; use a fresh Kaggle session"
subprocess.run(["git", "-C", str(REPO), "sparse-checkout", "init", "--cone"], check=True)
subprocess.run(["git", "-C", str(REPO), "sparse-checkout", "set", "code"], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", REPO_REF], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"], check=True)
CODE = REPO / "code"
assert (CODE / "train.py").is_file(), "Remote root must contain code/train.py"
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Code commit:", COMMIT)
if REPO_REF != COMMIT:
    print("For repeatable runs set REPO_REF to:", COMMIT)
sys.path.insert(0, str(CODE))


In [ ]:
from dataset import find_voc_root, inspect_dataset
from pretrained import EXPECTED_BYTES, EXPECTED_SHA256
from run_utils import file_sha256, write_json
INPUT = pathlib.Path("/kaggle/input")
DATA = find_voc_root(DATASET_DIR or INPUT)
if PRETRAINED_PATH:
    PRETRAIN = pathlib.Path(PRETRAINED_PATH)
    if not PRETRAIN.is_absolute():
        PRETRAIN = REPO / PRETRAIN
    PRETRAIN = PRETRAIN.resolve()
else:
    candidates = sorted(INPUT.rglob("extraction.conv.weights"))
    PRETRAIN = candidates[0] if len(candidates) == 1 else CODE / "pretrain/extraction.conv.weights"
    assert len(candidates) <= 1, "Multiple attached weights found; set PRETRAINED_PATH"
if not RESUME_PATH:
    assert PRETRAIN.is_file(), "Attach extraction.conv.weights or include code/pretrain in GitHub"
    assert PRETRAIN.stat().st_size == EXPECTED_BYTES
    assert file_sha256(PRETRAIN) == EXPECTED_SHA256, "Unexpected pretrained file"
OUT = WORK / RUN_NAME
ZIP = WORK / f"{RUN_NAME}.zip"
assert not OUT.exists(), "Use a new RUN_NAME; use RESUME_PATH to continue a previous run"
OUT.mkdir()
DATA_REPORT = inspect_dataset(DATA)
write_json(OUT / "dataset_inspection.json", DATA_REPORT)
print(DATA_REPORT)
print("Weights:", PRETRAIN, "Output:", ZIP)


In [ ]:
import json
import shutil
from export_results import export_results
CONFIG = CODE / "configs" / {"smoke": "smoke_ddp.json", "overfit": "overfit_ddp.json", "train": "voc2007_ddp.json"}[RUN_MODE]
if RESUME_PATH:
    saved = torch.load(RESUME_PATH, map_location="cpu", weights_only=True, mmap=True)
    config = saved["config"]
    assert saved.get("world_size", 1) == 2, "Resume requires a two-process DDP checkpoint"
    del saved
    CONFIG = OUT / "resume_config.json"
    write_json(CONFIG, config)
remaining_hours = MAX_HOURS - (time.monotonic() - SESSION_START) / 3600
assert remaining_hours > 0, "Setup exceeded the time budget"
command = [sys.executable, "-u", "-m", "torch.distributed.run",
           "--standalone", "--nnodes=1", "--nproc_per_node=2", str(CODE / "train.py"), "--config", str(CONFIG),
           "--data", str(DATA), "--output", str(OUT), "--device", "cuda",
           "--max-hours", str(remaining_hours), "--zip", str(ZIP)]
if RESUME_PATH:
    command += ["--resume", RESUME_PATH]
else:
    command += ["--pretrained", str(PRETRAIN)]
# A Python exception can still package diagnostic files; a forced session kill cannot.
try:
    subprocess.run(command, check=True)
finally:
    if not ZIP.exists():
        export_results(OUT, ZIP)
print("ZIP bytes:", ZIP.stat().st_size)


In [ ]:
from IPython.display import FileLink, display
manifest_path = OUT / "run_manifest.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    print("Status:", manifest["status"], "Completed epochs:", manifest.get("completed_epochs"))
metrics_path = OUT / "metrics.json"
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    print("Evaluation split:", metrics.get("split"), "mAP@0.5:", metrics["map50"])
    print("Classes with GT:", metrics["classes_with_gt"], "/", metrics["classes_total"])
display(FileLink(str(ZIP)))
# After Save Version -> Save & Run All, download ZIP from the notebook Output tab.
